In [ ]:
import duckdb

# 1. Connect to the database at the root project level
con = duckdb.connect('../amazon_sales_intelligence.db')

# 2. Try to run the blueprint. If it says tables exist, safely ignore and move on!
try:
    with open('schema.sql', 'r') as file:
        schema_sql = file.read()
    con.execute(schema_sql)
    print("Schema blueprints verified/created successfully.")
except duckdb.CatalogException as e:
    print(f"Note: Tables already structurally initialized ({e}). Proceeding to data check...")

# 3. Check if the fact table already has data loaded
row_check = con.execute("SELECT COUNT(*) FROM fact_product_metrics").fetchone()[0]

if row_check > 0:
    print(f"Tables already exist and data is already loaded! Found {row_check} product records.")
else:
    print("⏳ Tables empty. Streaming fresh CSV data into structured relational tables...")
    
    # Load Category Table
    con.execute("""
        INSERT INTO dim_category (category_id, category_main, category_sub) 
        SELECT category_id, category_main, category_sub 
        FROM read_csv_auto('../data/processed/dim_category.csv')
    """)

    # Load Product Table
    con.execute("""
        INSERT INTO dim_product (product_id, product_name, category_id, price_tier) 
        SELECT product_id, product_name, category_id, price_tier 
        FROM read_csv_auto('../data/processed/dim_product.csv')
    """)

    # Load Fact Table
    con.execute("""
        INSERT INTO fact_product_metrics (product_id, actual_price, discounted_price, discount_percentage, rating, rating_count, n_reviewers) 
        SELECT product_id, actual_price, discounted_price, discount_percentage, rating, rating_count, n_reviewers 
        FROM read_csv_auto('../data/processed/fact_product_metrics.csv')
    """)
    
    new_count = con.execute("SELECT COUNT(*) FROM fact_product_metrics").fetchone()[0]
    print(f"Success! Data loading complete. Loaded {new_count} records.")

# 4. Print visual summary confirmation
print("\nCurrent Active Database Layout")
print(con.execute("SHOW TABLES;").df())

# 5. Close connection explicitly to prevent persistent file locks
con.close()


Note: Tables already structurally initialized (Catalog Error: Table with name "dim_category" already exists!). Proceeding to data check...
Tables already exist and data is already loaded! Found 1350 product records.

Current Active Database Layout
                   name
0          dim_category
1           dim_product
2  fact_product_metrics


In [10]:
import duckdb

# 1. Re-open the database connection safely
con = duckdb.connect('../amazon_sales_intelligence.db')

# Every product should have a valid category (should return 0)
print(con.execute("""
    SELECT COUNT(*) AS orphaned_products 
    FROM dim_product p
    LEFT JOIN dim_category c ON p.category_id = c.category_id
    WHERE c.category_id IS NULL
""").df())


# Every fact row should have a matching product (should return 0)
print(con.execute("""
    SELECT COUNT(*) AS orphaned_metrics 
    FROM fact_product_metrics f
    LEFT JOIN dim_product p ON f.product_id = p.product_id
    WHERE p.product_id IS NULL
""").df())

# 3. Close connection explicitly to keep your database file clean
con.close()

   orphaned_products
0                  0
   orphaned_metrics
0                 0
